30/03 comparing leiden with hierarchical

In [ ]:
# ── CELL 1: Imports and Load ──────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import squidpy as sq
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import entropy as scipy_entropy
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/"
           "clustering_comparison")
os.makedirs(OUT_DIR, exist_ok=True)

adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

print("Shape:", adata.shape)
print("Spatial coords zero check:",
      (adata.obsm['spatial'] == 0).all(axis=1).sum(), "zeros")
print("Existing cluster columns:",
      [c for c in adata.obs.columns
       if any(x in c for x in ['niche', 'alpha', 'hc', 'leiden'])])

# Get CLR matrix as dense array — used throughout
X = adata.X.toarray() if sp.issparse(adata.X) else adata.X
print("CLR matrix shape:", X.shape)

In [ ]:
# ── Leiden resolution sweep ───────────────────────────────────────────
# spatial_niche already exists at res=0.5
# Compute additional resolutions for fair comparison with HC sweep

leiden_resolutions = [0.3, 0.7, 0.8, 0.9]
existing_cols = adata.obs.columns.tolist()

# Check which ones need computing
to_compute = [r for r in leiden_resolutions
              if f'leiden_res{r}' not in existing_cols]

if len(to_compute) == 0:
    print("All Leiden resolutions already computed.")
else:
    print(f"Computing Leiden at resolutions: {to_compute}")
    # neighbors graph already saved in h5ad — no need to rerun
    for res in to_compute:
        sc.tl.leiden(
            adata,
            resolution=res,
            key_added=f'leiden_res{res}',
            flavor='igraph',
            n_iterations=2,
            directed=False,
            random_state=42
        )
        n = adata.obs[f'leiden_res{res}'].nunique()
        print(f"  leiden_res{res}: {n} niches")

# Report all Leiden cluster counts
print("\nLeiden resolution sweep summary:")
for res in [0.3, 0.5, 0.7, 0.8, 0.9]:
    col = 'spatial_niche' if res == 0.5 else f'leiden_res{res}'
    if col in adata.obs.columns:
        n = adata.obs[col].nunique()
        print(f"  res={res}: {n} niches ({col})")

In [ ]:
# ── CELL 2: Check hierarchical clustering results ────────────────────
# Hierarchical clustering was run as a SLURM job (run_hierarchical_clustering.py)
# This cell verifies the results are present before proceeding

hc_cols = [c for c in adata.obs.columns if 'hc_ward' in c]
print("Hierarchical cluster columns found:", hc_cols)

if len(hc_cols) == 0:
    print("\nWARNING: No hierarchical cluster columns found.")
    print("Check if the SLURM job has finished:")
    print("  squeue -u YOUR_USERNAME")
    print("  cat hierarchical_clustering_[jobid].log")
else:
    print("\nSpot count distribution per k:")
    for col in sorted(hc_cols):
        counts = adata.obs[col].value_counts()
        max_pct = counts.max() / len(adata) * 100
        min_pct = counts.min() / len(adata) * 100
        print(f"  {col}: {counts.shape[0]} niches, "
              f"largest={max_pct:.1f}%, smallest={min_pct:.1f}%")
    print("\nReady to proceed.")

In [ ]:
# ── CELL 2b: Moran's I on continuous CLR values ──────────────────────
import squidpy as sq

# ── Part 1: Single slide inspection ──────────────────────────────────
print("Single slide Moran's I inspection...")
slide_id = 'Patient_2_H3_1'
adata_test = adata[adata.obs['slide_id'] == slide_id].copy()

# No key_added — use squidpy defaults throughout
sq.gr.spatial_neighbors(
    adata_test,
    coord_type='generic',
    n_neighs=6
)

# Check what key was created
print("obsp keys:", list(adata_test.obsp.keys()))

# Use default connectivity key
sq.gr.spatial_autocorr(
    adata_test,
    mode='moran'
)

print(f"\nMoran's I per cell type — {slide_id}")
print("(Higher = more spatially clustered)\n")
morans_slide = adata_test.uns['moranI'][['I', 'pval_norm']].copy()
morans_slide = morans_slide.sort_values('I', ascending=False)
print(morans_slide.to_string())

# ── Part 2: Global Moran's I across all slides ────────────────────────
print("\n\nComputing global Moran's I across all 38 slides...")

def compute_morans_i_continuous(adata, n_neighs=6):
    all_morans = []
    slides = adata.obs['slide_id'].unique()

    for i, slide_id in enumerate(slides):
        mask = adata.obs['slide_id'] == slide_id
        adata_slide = adata[mask].copy()

        if adata_slide.n_obs < 50:
            continue

        # No key_added — consistent with Part 1
        sq.gr.spatial_neighbors(
            adata_slide,
            coord_type='generic',
            n_neighs=n_neighs
        )

        # No connectivity_key — use squidpy default
        sq.gr.spatial_autocorr(
            adata_slide,
            mode='moran'
        )

        mean_I = adata_slide.uns['moranI']['I'].mean()
        all_morans.append(mean_I)

        if (i + 1) % 10 == 0:
            print(f"  Processed {i+1}/{len(slides)} slides...")

    return np.mean(all_morans), np.std(all_morans)

mean_I, std_I = compute_morans_i_continuous(adata)

print(f"\nGlobal mean Moran's I: {mean_I:.4f} ± {std_I:.4f}")

if mean_I > 0.3:
    print("  Strong spatial autocorrelation — justifies spatially-aware clustering.")
elif mean_I > 0.1:
    print("  Moderate spatial autocorrelation — some spatial structure present.")
else:
    print("  Weak spatial autocorrelation — compositions relatively uniform.")

print(f"\nThesis statement:")
print(f"  'Following Hirz et al. 2023, Moran's I computed on continuous")
print(f"  cell type abundance scores confirmed significant spatial")
print(f"  autocorrelation of compositional data (mean I = {mean_I:.4f}),")
print(f"  justifying spatially-aware clustering frameworks.'")

In [ ]:
# ── CELL 3: Dendrogram ───────────────────────────────────────────────
# Full dendrogram cannot be plotted for 101k spots
# Subsample 5000 spots and plot truncated dendrogram
# The dendrogram shows natural cluster boundaries — 
# large vertical gaps indicate stable cut points

print("Computing dendrogram on 5000-spot subsample...")

np.random.seed(42)
idx_sub = np.random.choice(len(adata), size=5000, replace=False)
X_sub = X[idx_sub]

# Compute linkage matrix
Z = linkage(X_sub, method='ward', metric='euclidean')
print("Linkage matrix computed.")

# Plot truncated dendrogram
fig, ax = plt.subplots(figsize=(14, 7))

dendrogram(
    Z,
    truncate_mode='lastp',   # show only the last p merges
    p=30,                    # show top 30 merges
    leaf_rotation=90,
    leaf_font_size=9,
    ax=ax,
    color_threshold=0.7 * max(Z[:, 2]),
    above_threshold_color='grey'
)

ax.set_title(
    'Hierarchical Clustering Dendrogram (Ward linkage)\n'
    'Subsample: 5,000 spots | Truncated to top 30 merges\n'
    'Large vertical gaps = natural cluster boundaries',
    fontsize=11
)
ax.set_xlabel('Cluster (spot count in brackets)', fontsize=10)
ax.set_ylabel('Ward Distance', fontsize=10)

# Add horizontal lines at distances corresponding to k=8,9,10,11
# to show where cuts would be made
for k, color, ls in [(8,'red','--'), (10,'blue','--'), (12,'green','--')]:
    # Find the distance threshold for k clusters
    # The last (n-k) merges give us k clusters
    # So threshold is between merge n-k and n-k+1
    n = len(X_sub)
    threshold = Z[-(k-1), 2]
    ax.axhline(y=threshold, color=color, linestyle=ls, alpha=0.7,
               label=f'k={k} cut')

ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'dendrogram_ward.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dendrogram_ward.png")

In [ ]:
# ── CELL 4: Silhouette Scores ────────────────────────────────────────
# Silhouette score measures how well-separated clusters are
# Range: -1 (wrong cluster) to +1 (perfect separation)
# Computed on subsample of 15,000 spots for speed

print("Computing silhouette scores...")
print("Using subsample of 15,000 spots for speed\n")

np.random.seed(42)
sil_idx = np.random.choice(len(adata), size=15000, replace=False)
X_sil = X[sil_idx]

# Define all clustering solutions to compare
# In cluster_cols dictionary in Cell 4
cluster_cols = {
    # Leiden — composition only, resolution sweep
    'Leiden res=0.3': 'leiden_res0.3',
    'Leiden res=0.5': 'spatial_niche',       # already exists
    'Leiden res=0.7': 'leiden_res0.7',
    'Leiden res=0.8': 'leiden_res0.8',
    'Leiden res=0.9': 'leiden_res0.9',
    # Hierarchical Ward — k sweep
    'HC Ward k=7':  'hc_ward_k7',
    'HC Ward k=8':  'hc_ward_k8',
    'HC Ward k=9':  'hc_ward_k9',
    'HC Ward k=10': 'hc_ward_k10',
    'HC Ward k=11': 'hc_ward_k11',
    'HC Ward k=12': 'hc_ward_k12',
}

sil_results = []

for name, col in cluster_cols.items():
    if col not in adata.obs.columns:
        print(f"  Skipping {col} — not found")
        continue

    labels_sil = adata.obs[col].values[sil_idx]
    n_unique = len(np.unique(labels_sil))

    if n_unique < 2:
        print(f"  Skipping {name} — only {n_unique} cluster")
        continue

    sil = silhouette_score(X_sil, labels_sil, metric='euclidean')
    n_clusters = adata.obs[col].nunique()
    counts = adata.obs[col].value_counts()
    max_pct = counts.max() / len(adata) * 100

    sil_results.append({
        'Method': name,
        'N clusters': n_clusters,
        'Silhouette': round(sil, 4),
        'Largest niche (%)': round(max_pct, 1)
    })
    print(f"  {name}: silhouette={sil:.4f}, n={n_clusters}")

sil_df = pd.DataFrame(sil_results)
print("\n── Silhouette Summary ──")
print(sil_df.sort_values('Silhouette', ascending=False).to_string(index=False))

# Plot silhouette scores
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3' if 'Leiden' in m else '#FF9800'
          for m in sil_df['Method']]
bars = ax.barh(sil_df['Method'], sil_df['Silhouette'],
               color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Silhouette Score', fontsize=11)
ax.set_title('Cluster Quality: Silhouette Score\n'
             'Blue = Leiden | Orange = Hierarchical',
             fontsize=11)
for bar, val in zip(bars, sil_df['Silhouette']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'silhouette_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: silhouette_comparison.png")

In [ ]:
# ── CELL 5: Spatial Coherence ────────────────────────────────────────
# Spatial Homophily — fraction of edges connecting same-niche spots
# Conceptually clean for categorical data but biased toward
# spatially-constrained methods


def compute_spatial_coherence(adata, cluster_key, n_neighs=6):
    homophily_scores = []
    slides = adata.obs['slide_id'].unique()

    for i, slide_id in enumerate(slides):
        mask = adata.obs['slide_id'] == slide_id
        adata_slide = adata[mask].copy()

        if adata_slide.n_obs < 50:
            continue

        sq.gr.spatial_neighbors(
            adata_slide,
            coord_type='generic',
            n_neighs=n_neighs,
            key_added='spatial_neighbors'
        )

        adj = adata_slide.obsp['spatial_neighbors_connectivities']
        sources, targets = adj.nonzero()
        labels = adata_slide.obs[cluster_key].values

        # Homophily
        same = (labels[sources] == labels[targets]).sum()
        homophily = same / len(sources)
        homophily_scores.append(homophily)

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(slides)} slides processed...")

    return np.mean(homophily_scores), np.std(homophily_scores)

spatial_results = []

for name, col in cluster_cols.items():
    if col not in adata.obs.columns:
        print(f"Skipping {col} — not found")
        continue

    print(f"Computing spatial coherence for: {name}")
    mean_H, std_H = compute_spatial_coherence(adata, col)

    spatial_results.append({
        'Method': name,
        'N clusters': adata.obs[col].nunique(),
        'Homophily (mean)': round(mean_H, 4),
        'Homophily (std)': round(std_H, 4)
    })
    print(f"  Homophily: {mean_H:.4f} ± {std_H:.4f}")

spatial_df = pd.DataFrame(spatial_results)
print("\n── Spatial Coherence Summary ──")
print(spatial_df.sort_values('Homophily (mean)',
                              ascending=False).to_string(index=False))

In [ ]:
# ── CELL 6: Sample Specificity (Entropy) ────────────────────────────
# Measures how well-mixed slides/patients are within each cluster
# High entropy = cluster shared across many slides = biological signal
# Low entropy = cluster dominated by one slide = possible artefact
# Computed at two levels: by slide_id AND by patient

print("Computing sample specificity (entropy)...")

def compute_entropy_scores(adata, cluster_key, groupby_col):
    n_groups = adata.obs[groupby_col].nunique()
    H_max = np.log2(n_groups) if n_groups > 1 else 1

    per_cluster = []
    weights = []

    for cluster in adata.obs[cluster_key].unique():
        mask = adata.obs[cluster_key] == cluster
        group_counts = adata.obs.loc[mask, groupby_col].value_counts()
        props = group_counts / group_counts.sum()
        H = scipy_entropy(props, base=2)
        H_norm = H / H_max
        per_cluster.append(H_norm)
        weights.append(mask.sum())

    weighted_mean = np.average(per_cluster, weights=weights)
    n_low = sum(e < 0.7 for e in per_cluster)
    return weighted_mean, n_low, per_cluster

entropy_results = []

for name, col in cluster_cols.items():
    if col not in adata.obs.columns:
        continue

    # By slide
    mean_slide, n_low_slide, _ = compute_entropy_scores(
        adata, col, 'slide_id')
    # By patient
    mean_patient, n_low_patient, _ = compute_entropy_scores(
        adata, col, 'patient')

    n_clusters = adata.obs[col].nunique()
    entropy_results.append({
        'Method': name,
        'N clusters': n_clusters,
        'Mean entropy (by slide)': round(mean_slide, 4),
        'Low-entropy clusters (slide)': n_low_slide,
        'Mean entropy (by patient)': round(mean_patient, 4),
        'Low-entropy clusters (patient)': n_low_patient,
    })
    print(f"{name}:")
    print(f"  By slide:   mean={mean_slide:.4f}, "
          f"low-entropy clusters={n_low_slide}/{n_clusters}")
    print(f"  By patient: mean={mean_patient:.4f}, "
          f"low-entropy clusters={n_low_patient}/{n_clusters}")

entropy_df = pd.DataFrame(entropy_results)

# Plot entropy comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col_name, title in zip(
    axes,
    ['Mean entropy (by slide)', 'Mean entropy (by patient)'],
    ['Sample Specificity by Slide', 'Sample Specificity by Patient']
):
    colors = ['#2196F3' if 'Leiden' in m else '#FF9800'
              for m in entropy_df['Method']]
    ax.barh(entropy_df['Method'], entropy_df[col_name],
            color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.axvline(x=0.7, color='red', linestyle='--',
               label='Threshold (0.7)', alpha=0.8)
    ax.set_xlabel('Normalised Entropy', fontsize=10)
    ax.set_title(f'{title}\nBlue=Leiden | Orange=HC | Red line=0.7 threshold',
                 fontsize=10)
    ax.set_xlim(0, 1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'entropy_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: entropy_comparison.png")

In [ ]:
# ── CELL 7: Summary Comparison Table ────────────────────────────────
# Combine all metrics into one table for supervisor presentation
# 
# Three per-method metrics:
#   Silhouette — cluster compactness in compositional space
#   Homophily  — spatial coherence of cluster assignments
#   Entropy    — patient/slide mixing within clusters
#
# One global data metric (not per-method):
#   Moran's I  — intrinsic spatial structure of CLR compositions
#                computed following Hirz et al. 2023

print("Building summary comparison table...")

# Start with silhouette
summary = sil_df.copy()

# Merge homophily (replaces Moran's I as per-method spatial metric)
if 'spatial_df' in dir() and spatial_df is not None and len(spatial_df) > 0:
    summary = summary.merge(
        spatial_df[['Method', 'Homophily (mean)']],
        on='Method', how='left'
    )

# Merge entropy
if 'entropy_df' in dir() and entropy_df is not None and len(entropy_df) > 0:
    summary = summary.merge(
        entropy_df[['Method',
                    'Mean entropy (by slide)',
                    'Low-entropy clusters (slide)']],
        on='Method', how='left'
    )

print("\n" + "="*80)
print("COMPLETE CLUSTERING COMPARISON")
print("="*80)
print(summary.to_string(index=False))

# Add global Moran's I as contextual note
print("\n" + "="*80)
print("GLOBAL DATA CONTEXT (not per-method)")
print("="*80)
try:
    print(f"Mean Moran's I across all slides: {mean_I:.4f} ± {std_I:.4f}")
    print("(Computed on continuous CLR values following Hirz et al. 2023)")
    print("This confirms intrinsic spatial structure in the data,")
    print("justifying spatially-aware clustering approaches.")
except NameError:
    print("mean_I not computed yet — run Cell 2b first")

# Heatmap visualisation of the comparison table
numeric_cols = [c for c in summary.columns
                if c not in ['Method', 'N clusters',
                             'Low-entropy clusters (slide)',
                             'Largest niche (%)']]

if len(numeric_cols) > 0:
    import seaborn as sns
    import matplotlib.pyplot as plt

    summary_norm = summary[['Method']].copy()
    for col in numeric_cols:
        col_vals = summary[col].fillna(0)
        col_range = col_vals.max() - col_vals.min()
        if col_range > 0:
            summary_norm[col] = (col_vals - col_vals.min()) / col_range
        else:
            summary_norm[col] = 0.5

    summary_norm_plot = summary_norm.set_index('Method')

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(
        summary_norm_plot,
        cmap='RdYlGn',
        vmin=0, vmax=1,
        annot=summary[numeric_cols].round(4).values,
        fmt='',
        ax=ax,
        linewidths=0.5,
        cbar_kws={'label': 'Normalised score (higher=better)'}
    )
    ax.set_title(
        'Clustering Method Comparison\n'
        'All metrics normalised 0-1 | Green=better | Red=worse\n'
        f'Global Moran\'s I = {mean_I:.4f} (data spatial structure)',
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'clustering_comparison_table.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: clustering_comparison_table.png")

In [ ]:
# ── CELL 8: Niche Heatmap for Best Hierarchical k ───────────────────
# Identify the best hierarchical k based on metrics above
# Then run full niche characterisation to compare biology with Leiden

# Set this based on Cell 4-6 results
# Start with k=10 to match Leiden niche count
best_hc_col = 'hc_ward_k10'  # adjust after seeing metrics

print(f"Running niche characterisation for {best_hc_col}...")

def niche_heatmap(adata, cluster_key, title_suffix="", save_dir=OUT_DIR):
    X_df = pd.DataFrame(
        adata.X.toarray() if sp.issparse(adata.X) else adata.X,
        columns=adata.var_names,
        index=adata.obs_names
    )
    X_df['niche'] = adata.obs[cluster_key].values
    mean_profile = X_df.groupby('niche').mean()

    study_frac = (
        adata.obs.groupby(cluster_key)['study']
        .value_counts(normalize=True)
        .unstack().fillna(0)
    )
    zone_frac = (
        adata.obs.groupby(cluster_key)['zone']
        .value_counts(normalize=True)
        .unstack().fillna(0)
    )

    fig, axes = plt.subplots(
        1, 3, figsize=(22, max(6, len(mean_profile) * 0.6)),
        gridspec_kw={'width_ratios': [4, 1, 1]}
    )
    sns.heatmap(mean_profile, cmap='RdBu_r', center=0,
                ax=axes[0], cbar_kws={'label': 'CLR abundance'},
                linewidths=0.3)
    axes[0].set_title(f'Cell type profile — {title_suffix}')

    sns.heatmap(study_frac, cmap='Oranges', vmin=0, vmax=1,
                ax=axes[1], cbar_kws={'label': 'Fraction'})
    axes[1].set_title('Study composition')
    axes[1].set_ylabel('')

    sns.heatmap(zone_frac, cmap='Greens', vmin=0, vmax=1,
                ax=axes[2], cbar_kws={'label': 'Fraction'})
    axes[2].set_title('Zone composition')
    axes[2].set_ylabel('')

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'heatmap_{cluster_key}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    return mean_profile

# Run heatmap for best HC and best Leiden side by side
for col, title in [
    (best_hc_col, f'Hierarchical Ward {best_hc_col}'),
    ('spatial_niche', 'Leiden comp-only res=0.5'),
    ('niche_alpha_0.9_res0.8', 'Leiden joint graph res=0.8')
]:
    if col in adata.obs.columns:
        niche_heatmap(adata, col, title_suffix=title)
    else:
        print(f"Skipping {col} — not found")

In [ ]:
# ── CELL 9: Tissue Status and ISUP Grade for Best HC ────────────────

for col, label in [(best_hc_col, 'HC Ward')]:
    if col not in adata.obs.columns:
        continue

    # Tissue status
    ct = pd.crosstab(
        adata.obs[col],
        adata.obs['tissue_status'],
        normalize='index'
    )
    fig, ax = plt.subplots(figsize=(8, max(5, len(ct) * 0.5)))
    sns.heatmap(ct, cmap='RdYlGn', vmin=0, vmax=1,
                annot=True, fmt='.2f', ax=ax)
    ax.set_title(f'Tissue status per niche — {col}')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'tissue_status_{col}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    # ISUP grade — Erickson only
    erickson_mask = adata.obs['study'] == 'Erickson_2022'
    ct_isup = pd.crosstab(
        adata.obs[erickson_mask][col],
        adata.obs[erickson_mask]['isup_grade'],
        normalize='index'
    )
    fig, ax = plt.subplots(figsize=(8, max(5, len(ct_isup) * 0.5)))
    sns.heatmap(ct_isup, cmap='Reds', annot=True, fmt='.2f', ax=ax)
    ax.set_title(f'ISUP grade (Erickson only) — {col}')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'isup_{col}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

Now doing HC with the code that Gillaume send me (doesnt use CLR transformation)

In [ ]:
import scanpy as sc
adata_ref = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/reference_model/c2l_curated_reference.h5ad"
)
# Get all studies with cell counts and DOIs
study_info = adata_ref.obs.groupby('study_id').agg(
    n_cells=('study_id', 'count'),
    study_name=('study_name', 'first'),
    study_doi=('study_doi', 'first'),
    study_authors=('study_authors', 'first')
).reset_index()

study_info = study_info.sort_values('n_cells', ascending=False)

for _, row in study_info.iterrows():
    print(f"\n{row['study_id']}")
    print(f"  n_cells: {row['n_cells']}")
    print(f"  doi: {row['study_doi']}")
    print(f"  name: {row['study_name'][:80]}")